# 01 — Selected dataset setup and inspection
Downloads and inspects the dataset selected once in `src/notebook_config.py`. Test clips are counted but never decoded for evaluation.

In [2]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path('/content/phd_research')
if not REPO_ROOT.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/sanelehlabisa/phd_research.git', str(REPO_ROOT)], check=True)
else:
    checkout_dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
    if checkout_dirty:
        print('Existing Colab checkout has local changes; preserving them and skipping update.')
    else:
        subprocess.run(['git', 'pull', '--ff-only', 'origin', 'master'], cwd=REPO_ROOT, check=True)
IMPLEMENTATION_ROOT = REPO_ROOT / 'papers/001-journal-abnormal-activity-recognition/implementation'
bootstrap = subprocess.run([sys.executable, str(IMPLEMENTATION_ROOT / 'src/colab_bootstrap.py'), str(IMPLEMENTATION_ROOT / 'requirements.txt')])
if bootstrap.returncode == 75:
    raise SystemExit('Restart the Colab runtime, reconnect to the A100, and rerun from the top.')
bootstrap.check_returncode()
if str(IMPLEMENTATION_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION_ROOT))


CalledProcessError: Command '['/usr/bin/python3', '/content/phd_research/papers/001-journal-abnormal-activity-recognition/implementation/src/colab_bootstrap.py', '/content/phd_research/papers/001-journal-abnormal-activity-recognition/implementation/requirements.txt']' returned non-zero exit status 2.

In [ ]:
from src.notebook_utils import printable, prepare_selected_splits, validate_runtime
print(printable(validate_runtime(require_cuda=True)))


In [ ]:
prepared = prepare_selected_splits(IMPLEMENTATION_ROOT)
inventory_summary = {key: value for key, value in prepared['inventory'].items() if key != 'records'}
print(printable({
    'dataset': prepared['specification'],
    'inventory': inventory_summary,
    'split': prepared['split'],
    'test_count_locked': prepared['test_count_locked'],
}))


In [ ]:
training_index = prepared['train'].indices[0]
clip, label = prepared['dataset'][training_index]
print({'partition': 'train', 'shape': tuple(clip.shape), 'label': prepared['dataset'].class_names[label]})
